In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from src.features.new_features import NewFeature, TimeFeatureStrategy, AgeCategoryFeatureStrategy, \
    HourCategoryFeatureStrategy
from src.models.model import RegressionLogisticModel, RandomForestClassifierModel

In [2]:
df_transactions = pd.read_csv('../data/raw/transactions.csv', encoding='utf-8')
df_customers = pd.read_csv('../data/raw/customers.csv', encoding='utf-8')
df = pd.merge(
    df_transactions,
    df_customers,
    left_on='sender_id',
    right_on='customer_id',
    how='left'
)

In [3]:
features = [
    TimeFeatureStrategy(),
    AgeCategoryFeatureStrategy(),
    HourCategoryFeatureStrategy()
]

for f in features:
    processor = NewFeature(f)
    df_feature = processor.apply(df)

In [4]:
df_feature.columns

Index(['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'amount',
       'device_type', 'fraud', 'customer_id', 'cpf', 'age', 'gender',
       'pix_key', 'account_type', 'city', 'hour_date', 'minute_date',
       'age_category', 'hour_category'],
      dtype='object')

In [5]:
df_transformed = df_feature.drop(
    columns=['transaction_id', 'timestamp', 'sender_id', 'receiver_id', 'customer_id', 'cpf', 'pix_key', 'hour_date',
             'minute_date'])

In [6]:
X = df_transformed.drop(columns=['fraud'])
y = df_transformed['fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
print(f'Target Treino {y_train.value_counts()[1]}, Target teste {y_test.value_counts()[1]}')
print(f'Target Treino {(y_train.value_counts()[1] / y_train.value_counts()[0]) * 100:.2f}%')
print(f'Target Teste {(y_test.value_counts()[1] / y_test.value_counts()[0]) * 100:.2f}%')

Target Treino 892, Target teste 223
Target Treino 8.03%
Target Teste 8.03%


In [8]:
list_onehot = ['hour_category', 'age_category', 'account_type', 'gender', 'device_type']
list_num = ['amount', 'age']

cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
num_pipeline = Pipeline([
    ('num', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, list_num),
    ('cat', cat_pipeline, list_onehot),
])


In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

num_cols = list_num
cat_cols = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(list_onehot)

all_cols = list(num_cols) + list(cat_cols)

X_train_df = pd.DataFrame(X_train_processed, columns=all_cols)

In [10]:
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)
X_resampled, y_resampled = smote.fit_resample(X_train_df, y_train)
print(y_resampled.value_counts())

fraud
0    11108
1    11108
Name: count, dtype: int64


In [11]:
models = [
    RegressionLogisticModel(),
    RandomForestClassifierModel()
]

param_grids = [
    {  #LogisticRegression
        'classifier__C': [0.1, 1, 10],
        'classifier__penalty': ['l2'],
        'classifier__solver': ['lbfgs']
    },
    {  #RandomForest
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10, 20]
    }
]


In [12]:
results = []

for model, param_grid in zip(models, param_grids):
    pipeline_model = ImbPipeline([
        ('classifier', model.model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline_model,
        param_grid=param_grid,
        cv=3,
        scoring='f1',
        n_jobs=-1
    )

    grid_search.fit(X_resampled, y_resampled)

    results.append({
        'model': model.__class__.__name__,
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'best_model': grid_search.best_estimator_
    })


In [13]:
for r in results:
    print(f"Modelo: {r['model']}")
    print("Melhores hiperparâmetros:", r['best_params'])
    print("F1 médio no CV:", r['best_score'])

    y_pred = r['best_model'].predict(X_test_processed)

    print("Relatório de classificação:")
    print(classification_report(y_test, y_pred))

    print("Matriz de confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("-" * 50)

Modelo: RegressionLogisticModel
Melhores hiperparâmetros: {'classifier__C': 0.1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
F1 médio no CV: 0.5392315596202475
Relatório de classificação:
              precision    recall  f1-score   support

           0       0.95      0.76      0.84      2777
           1       0.14      0.50      0.22       223

    accuracy                           0.74      3000
   macro avg       0.55      0.63      0.53      3000
weighted avg       0.89      0.74      0.80      3000

Matriz de confusão:
[[2102  675]
 [ 111  112]]
--------------------------------------------------
Modelo: RandomForestClassifierModel
Melhores hiperparâmetros: {'classifier__max_depth': None, 'classifier__n_estimators': 200}
F1 médio no CV: 0.8790971780082429
Relatório de classificação:
              precision    recall  f1-score   support

           0       0.96      0.91      0.93      2777
           1       0.29      0.47      0.36       223

    accuracy     

C:\Users\compu\anaconda3\envs\env-conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\compu\anaconda3\envs\env-conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
